# Sentinel-1 and Sentinel-2 parcel zonal statistics

Configure and run `SatelliteZonalStats`. openEO creates one cloud-masked, indexed monthly NetCDF cube per spatial batch. IQR cleaning, null filling, and parcel statistics run locally, producing one ML-ready row per parcel and month.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Run this notebook from within the satellites repository.")
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import geopandas as gpd
from satellite_extract_data.zonal_stats_job_manager import JobManagerSatelliteZonalStats, compute_tile_buffer_metres, compute_tile_width
from common_libraries.io_library import read_data, write_data
import common_libraries.geom_library as geom_l

## Inputs

Processing order:

`Sentinel-2 L2A`
`acquisition cloud mask`
`acquisition pixel indices`
`monthly pixel median`
`monthly per-variable cube IQR outlier removal`
`local null filling`
`parcel statistics`
`compact table`

Temporal filling is past-only so later observations cannot leak into earlier feature periods.

In [ ]:
WORK_PATH = Path("C:/work_dir")
PARCELS_PATH = WORK_PATH / "gr_parcels" / "all_predicted_parcels_labels.parquet"

TEST_AREA_PARCELS_PATH = WORK_PATH / "gr_parcels" / "kozani_parcels.gpkg"

In [ ]:
# gdf_parcels = read_data(str(PARCELS_PATH))
# gdf_parcels = geom_l.return_valid_geometries(gdf_parcels)
# gdf_parcels = gdf_parcels.to_crs("EPSG:2100")

# gdf_plesio = read_data(str(TEST_AREA_PLESION_PATH))
# gdf_plesio = geom_l.return_valid_geometries(gdf_plesio)
# gdf_plesio = gdf_plesio.to_crs("EPSG:2100")
# gdf_plesio["geometry"] = gdf_plesio["geometry"].buffer(10000, cap_style=3)

# gdf_parcels = gpd.sjoin(gdf_parcels, gdf_plesio[["geometry"]], how="inner", predicate="intersects")
# gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)
# gdf_parcels["parcel_code"] = gdf_parcels["parcel_code"].astype(str)
# gdf_parcels = gdf_parcels.rename(columns={"initial_label_code": "label"})
# gdf_parcels = gdf_parcels[["parcel_code", "label", "geometry"]].copy()
# write_data(gdf_parcels, str(TEST_AREA_PARCELS_PATH))


In [ ]:
gdf_parcels = read_data(str(TEST_AREA_PARCELS_PATH))
gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)
print(len(gdf_parcels), "parcels in test area")
gdf_parcels.head(3)

In [ ]:
class_column = "label"
parcel_id_column = "parcel_code"

In [ ]:
START_DATE = "2023-10-01"
END_DATE = "2024-09-30"
RUN_IDENTIFIER = "kozani-2023-2024"  # Change this for each named run.
OUTPUT_DIR = WORK_PATH / "kozani_parcels"
WORKING_EPSG = 32634

CALCULATE_SENTINEL2_INDICES = True
SENTINEL2_INDICES = ["EVI", "NDMI", "NDRE", "NDVI", "NDWI",  "SAVI"]
SENTINEL2_BANDS = ["B02", "B03", "B04", "B05", "B08", "B11", "B12"]
SENTINEL1_BANDS = [] # ["VV", "VH"]

# Mean is mandatory; percentiles enable derived IQR and robust skewness.
SPATIAL_STATISTICS = [
    "mean",
    "median",
    "sd",
    "min",
    "max",
    # "count",
    "p10",
    "p25",
    "p75",
    "p90",
]

INTERPOLATION_METHOD = "nearest"
INTERPOLATION_MAX_DISTANCE_IN_METERS = 500 

# IQR outliers become null, then nulls are filled.
REMOVE_OUTLIERS = True
IQR_QUANTILES = (0.1, 0.9)
IQR_MULTIPLIER = 1.5
FILL_NULLS = True
IQR_MIN_VALID_PIXELS = 20
TEMPORAL_FILL_MODE = "bidirectional" # or past_only
MINIMUM_PARCEL_PIXELS = 3
MINIMUM_OBSERVED_FRACTION_FOR_FILL = 0.20

# Each worker loads one complete monthly cube; use one unless RAM has been sized for parallel cubes.
LOCAL_BATCH_WORKERS = 1

gdf_parcels = gdf_parcels.to_crs(WORKING_EPSG)

## Tile-size recommendation

This local comparison selects the smallest candidate grid whose buffered ownership tiles fully cover every parcel.

In [ ]:
TILE_SIZE_METRES = compute_tile_width(gdf_parcels, parcel_id_col=parcel_id_column, working_epsg=WORKING_EPSG)
TILE_BUFFER_METRES = compute_tile_buffer_metres(gdf_parcels, WORKING_EPSG)

## Run extraction

In [ ]:
TILE_SIZE_METRES = 30_000
TILE_BUFFER_METRES = 3_000

In [ ]:
credentials = [
    entry.strip().split(",", maxsplit=1)
    for entry in os.environ["OPENEO_USERS"].split(";")
    if entry.strip()
]

username, password = credentials[0]  # First user
username

In [ ]:
extractor = JobManagerSatelliteZonalStats(
    parcels=gdf_parcels,
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
    working_epsg=WORKING_EPSG,
    
    openeo_username=username,
    openeo_password=password,
    openeo_parallel_jobs=2,
    tile_size_metres=TILE_SIZE_METRES,
    tile_buffer_metres=TILE_BUFFER_METRES,
    job_poll_seconds=30,
    run_identifier=RUN_IDENTIFIER,
        
    parcel_id_field=parcel_id_column,
    sentinel2_bands=SENTINEL2_BANDS,
    calculate_sentinel2_indices=CALCULATE_SENTINEL2_INDICES,
    sentinel2_indices=SENTINEL2_INDICES,
    sentinel1_bands=SENTINEL1_BANDS,
    spatial_statistics=SPATIAL_STATISTICS,
    
    minimum_parcel_pixels=MINIMUM_PARCEL_PIXELS,
    minimum_observed_fraction_for_fill=MINIMUM_OBSERVED_FRACTION_FOR_FILL,
        
    temporal_fill_mode=TEMPORAL_FILL_MODE,
    interpolation_method=INTERPOLATION_METHOD,
    interpolation_max_distance_in_meters=INTERPOLATION_MAX_DISTANCE_IN_METERS,
    
    remove_outliers=REMOVE_OUTLIERS,
    iqr_quantiles=IQR_QUANTILES,
    iqr_multiplier=IQR_MULTIPLIER,
    fill_nulls=FILL_NULLS,
    iqr_min_valid_pixels=IQR_MIN_VALID_PIXELS,
    
    batch_workers=LOCAL_BATCH_WORKERS,
)
results = extractor.run()
results.head()

## Result analysis

In [ ]:
# Reload the final table in a later session if needed.
# import pandas as pd
# results = pd.read_parquet(OUTPUT_DIR / "satellite_temporal_parcel_statistics.parquet")
# results

In [ ]:
import numpy as np
import pandas as pd

from data_analysis.eda import EDAConfig, EDAPipeline, run_eda
import webbrowser

In [ ]:
tmp = results.copy()
tmp = tmp.drop(columns=["geometry"], errors="ignore")
tmp[parcel_id_column] = tmp[parcel_id_column].astype(str)
parcel_attributes = gdf_parcels[[parcel_id_column, class_column, "geometry"]].copy()
parcel_attributes[parcel_id_column] = parcel_attributes[parcel_id_column].astype(str)
df_merged = parcel_attributes.merge(tmp, on=parcel_id_column, how="left")

write_data(df_merged, str(OUTPUT_DIR / "final_parcel_statistics.parquet"))
df_merged.head()

In [ ]:
try:
    base_columns = [parcel_id_column, class_column, "geometry"]
    import geopandas as gpd
    from shapely.geometry import Point

    EDA_PATH = OUTPUT_DIR / "results_eda"

    kwargs = {
        "max_columns_for_plots": 15,
        "max_pairplot_features": 15,
        "max_multivariate_features": 15,
        "id_column": "parcel_code",
    }
    
    geo_result = run_eda(
        df_merged[base_columns + [x for x in list(df_merged.columns) if "median" in x]],
        output_dir=str(EDA_PATH),
        target_column=class_column,
        report_title="Sample GeoDataFrame EDA",        
        **kwargs
    )
    html_path = geo_result["report_path"]
    webbrowser.open(html_path.as_uri())
except ImportError as exc:
    print(f"GeoDataFrame example skipped: {exc}")

In [ ]:
# Distribution of one feature over time, with an optional parcel overlay.
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

parcel_to_check = None # df["parcel_code"].iloc[0]  # Set to None to hide the parcel line.
date_column = "period_start"
show_outliers = False

for feature in [x for x in df_merged.select_dtypes(include="number").columns if x != class_column]:
    plot_data = df_merged[[date_column, parcel_id_column, class_column, feature]].copy()
    plot_data[date_column] = pd.to_datetime(plot_data[date_column], errors="coerce")
    plot_data[feature] = pd.to_numeric(plot_data[feature], errors="coerce")
    plot_data = plot_data.dropna(subset=[date_column, feature]).sort_values(date_column)
    plot_data["period_label"] = plot_data[date_column].dt.strftime("%Y-%m-%d")

    fig_width = max(20, plot_data["period_label"].nunique() * 0.7)
    fig, ax = plt.subplots(figsize=(fig_width, 6))

    if class_column is not None:
        sns.boxplot(data=plot_data, x="period_label", y=feature, color="lightsteelblue", hue=class_column, showfliers=show_outliers, ax=ax)
        sns.stripplot(data=plot_data, x="period_label", y=feature, color="darkblue", size=3, alpha=0.5, jitter=True, ax=ax)
    else:
        sns.boxplot(data=plot_data, x="period_label", y=feature, color="lightsteelblue", showfliers=show_outliers, ax=ax)

    if parcel_to_check is not None:
        parcel_data = plot_data.loc[plot_data["parcel_code"].eq(parcel_to_check)]
        sns.lineplot(
            data=parcel_data,
            x="period_label",
            y=feature,
            color="crimson",
            marker="o",
            linewidth=2,
            label=f"Parcel {parcel_to_check}",
            sort=False,
            ax=ax,
        )

    ax.set_xlabel("Period start")
    ax.set_ylabel(feature)
    ax.set_title(f"Distribution of {feature} over time")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_29516\2378601077.py:22: FutureWarning: 

Setting a gradient palette using color= is deprecated and will be removed in v0.14.0. Set `palette='dark:lightsteelblue'` for the same effect.

  sns.boxplot(data=plot_data, x="period_label", y=feature, color="lightsteelblue", hue=class_column, showfliers=show_outliers, ax=ax)
